In [ ]:
"""
Portfolio Multifractal Comparison
==================================
6-panel figure per portfolio + cross-portfolio scorecard.

Fixes applied:
  1. IndexError in portfolio_returns(): dates_i now aligns exactly with
     rets_i (both start after the first price point), so idx_map positions
     are always valid indices into rets_i.
  2. Expanded q-grid for MF-DFA to cover a wider alpha range:
     Q_VALS now runs from -40 to +40 (step 2), giving finer resolution
     of the singularity spectrum tails.
"""

import numpy as np
import yfinance as yf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
from scipy.signal import savgol_filter
import os, warnings

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# PORTFOLIOS
# ─────────────────────────────────────────────────────────────────────────────
PORTFOLIOS = {
    "Momentum":        ["NVDA", "TSLA",    "META",    "AMZN",    "NFLX"],
    "Defensive":       ["JNJ",  "KO",      "PG",      "WMT",     "PEP"],
    "Financials":      ["JPM",  "GS",      "BAC",     "MA",      "V"],
    "Commodities":     ["GC=F", "SI=F",    "CL=F",    "BTC-USD", "ETH-USD"],
    "Diversified":     ["AAPL", "JPM",     "JNJ",     "XOM",     "EWJ"],
    "High-Conviction": ["NVDA", "BTC-USD", "TSLA",    "ETH-USD", "META"],
}

# Pre-computed min Pearson r(q) between F_{+q} and F_{-q}
MIN_R = {
    "AAPL": 0.9700, "AMZN": 0.9744, "NVDA": 0.9706, "META": 0.9287,
    "TSLA": 0.9324, "NFLX": 0.9841,
    "JPM":  0.9842, "BAC":  0.9786, "GS":   0.9468,
    "V":    0.8369, "MA":   0.7516,
    "JNJ":  0.3982, "PG":   0.8195, "KO":   0.2383, "PEP": -0.1332,
    "WMT":  0.9644, "XOM":  0.9296,
    "GC=F": 0.9118, "SI=F": 0.8545, "CL=F": 0.7178,
    "BTC-USD": 0.9769, "ETH-USD": 0.9670,
    "EWJ":  0.9581,
}

START         = "2015-01-01"
END           = "2026-04-11"

# ── q-grid: step=5, range ±30, INCLUDING q=0 ─────────────────────────────────
# Three deliberate choices that match the better-spectrum script:
#   1. q=0 included  → geometric-mean formula captures the f(α) apex correctly
#   2. step=5 (not 2) → coarser spacing reduces finite-difference noise in
#      dτ/dq, giving a smoother α(q) and a cleaner f(α) parabola
#   3. range ±30      → wide enough to capture fat-tail scaling (crypto/momentum)
#      without pushing into regimes where rms^q overflows for large |q|
Q_VALS  = np.arange(-30, 31, 5, dtype=float)   # includes 0

# q-grid for Rényi partition function (also expanded)
Q_RENYI = np.concatenate([
    np.linspace(-20, -0.5, 20),
    np.linspace( 0.5,  20, 20)
])

POLY_ORD      = 1
OVERLAP       = 0.75
N_SCALES      = 30
SCALE_MIN     = 20
SCALE_MAX     = 800
REGIME_WINDOW = 120
VAR_WINDOW    = 60
CONFIDENCE    = 0.99
POSITION_SIZE = 100_000

OUT_DIR = "portfolio_mf_plots"
os.makedirs(OUT_DIR, exist_ok=True)

PORTFOLIO_COLORS = {
    "Momentum":        "#f85149",
    "Defensive":       "#3fb950",
    "Financials":      "#58a6ff",
    "Commodities":     "#e3b341",
    "Diversified":     "#bc8cff",
    "High-Conviction": "#ff7f0e",
}

TICKER_PALETTE = [
    "#e63946", "#2a9d8f", "#e9c46a", "#264653", "#a8dadc",
    "#457b9d", "#f4a261", "#6a4c93", "#80b918", "#f72585",
]

# ─────────────────────────────────────────────────────────────────────────────
# MF-DFA CORE
# ─────────────────────────────────────────────────────────────────────────────
def integrate(x):
    return np.cumsum(x - x.mean())


def _overlap_starts(N, s, overlap):
    step = max(1, int(s * (1.0 - overlap)))
    return np.arange(0, N - s + 1, step)


def _segment_rms_list(x, scale_min, scale_max, n_scales,
                      poly_ord=POLY_ORD, overlap=OVERLAP):
    y         = integrate(x)
    N         = len(y)
    scale_max = min(scale_max, N // 4)
    scale_min = max(scale_min, poly_ord + 2)
    if scale_max < scale_min:
        return np.array([]), []
    scales = np.unique(
        np.logspace(np.log10(scale_min), np.log10(scale_max), n_scales).astype(int)
    )
    scales = scales[scales >= scale_min]
    rms_per = []
    for s in scales:
        starts = _overlap_starts(N, s, overlap)
        if len(starts) < 4:
            rms_per.append(np.array([])); continue
        t, rms = np.arange(s), []
        for st in starts:
            seg  = y[st:st+s]
            coef = np.polyfit(t, seg, poly_ord)
            r    = np.sqrt(np.mean((seg - np.polyval(coef, t))**2))
            if r > 0:
                rms.append(r)
        rms_per.append(np.asarray(rms))
    return scales, rms_per


def mfdfa(x, q_vals, scale_min=SCALE_MIN, scale_max=SCALE_MAX,
          n_scales=N_SCALES, poly_ord=POLY_ORD, overlap=OVERLAP):
    scales, rms_per = _segment_rms_list(x, scale_min, scale_max,
                                         n_scales, poly_ord, overlap)
    if len(scales) == 0:
        return (np.array([]),
                np.full((len(q_vals), 0), np.nan),
                np.full(len(q_vals), np.nan))

    Fqs = np.full((len(q_vals), len(scales)), np.nan)
    for i, rms in enumerate(rms_per):
        if len(rms) == 0:
            continue
        for j, q in enumerate(q_vals):
            if q == 0:
                Fqs[j, i] = np.exp(0.5 * np.mean(np.log(rms**2)))
            else:
                Fqs[j, i] = (np.mean(rms**q))**(1.0/q)

    hq = np.full(len(q_vals), np.nan)
    for j in range(len(q_vals)):
        row  = Fqs[j]
        mask = np.isfinite(row) & (row > 0)
        if mask.sum() >= 4:
            hq[j] = np.polyfit(np.log(scales[mask]),
                                np.log(row[mask]), 1)[0]
    return scales, Fqs, hq


def multifractal_spectrum(q_vals, hq):
    tau_q = q_vals * hq - 1.0

    # With step=5 we get 13 q-values. window=5, polyorder=2 matches script 2
    # and avoids over-smoothing the apex while suppressing finite-diff noise.
    # Only smooth the valid (non-NaN) region to avoid edge contamination.
    n = len(tau_q)
    wl = min(5, n if n % 2 == 1 else n - 1)
    wl = max(wl, 3)
    valid_mask = np.isfinite(tau_q)
    tau_s = tau_q.copy()
    if valid_mask.sum() >= wl:
        tau_s[valid_mask] = savgol_filter(tau_q[valid_mask], wl, 2)

    alpha = np.gradient(tau_s, q_vals)
    D_a   = q_vals * alpha - tau_s
    return tau_q, alpha, D_a


def delta_alpha(alpha, D_a):
    valid = np.isfinite(alpha) & np.isfinite(D_a)
    if valid.sum() < 2:
        return np.nan
    return float(alpha[valid].max() - alpha[valid].min())


def hq_slope(q_vals, hq):
    valid = np.isfinite(hq)
    if valid.sum() < 4:
        return np.nan
    return float(np.mean(np.abs(np.gradient(hq[valid], q_vals[valid]))))


# ─────────────────────────────────────────────────────────────────────────────
# RÉNYI PARTITION FUNCTION
# ─────────────────────────────────────────────────────────────────────────────
def _log_chi_q(rms, q):
    if len(rms) < 3:
        return np.nan
    p = rms**2 / (rms**2).sum()
    if q == 1:
        p_s = p[p > 0]
        return float(-np.sum(p_s * np.log(p_s)))
    val = np.sum(p**q)
    return float(np.log(val)) if val > 0 else np.nan


def renyi_tau(x, q_vals, scale_min=SCALE_MIN, scale_max=SCALE_MAX,
              n_scales=N_SCALES, poly_ord=POLY_ORD, overlap=OVERLAP):
    scales, rms_per = _segment_rms_list(x, scale_min, scale_max,
                                         n_scales, poly_ord, overlap)
    if len(scales) == 0:
        return np.full(len(q_vals), np.nan)
    tau_r = np.full(len(q_vals), np.nan)
    for j, q in enumerate(q_vals):
        lchi = np.array([_log_chi_q(r, q) for r in rms_per])
        mask = np.isfinite(lchi) & \
               (np.array([len(r) for r in rms_per]) >= 3)
        if mask.sum() >= 4:
            tau_r[j] = np.polyfit(np.log(scales[mask]),
                                   lchi[mask], 1)[0]
    return tau_r


def renyi_V(q_vals, tau_r):
    valid = np.isfinite(tau_r)
    if valid.sum() < 6:
        return np.nan
    q_v, t_v = q_vals[valid], tau_r[valid]
    resid = t_v - np.polyval(np.polyfit(q_v, t_v, 1), q_v)
    return float(np.std(resid))


# ─────────────────────────────────────────────────────────────────────────────
# VaR
# ─────────────────────────────────────────────────────────────────────────────
def historical_var(returns, confidence=CONFIDENCE):
    return -np.percentile(returns, (1.0 - confidence) * 100.0)


def renyi_adjusted_var(ret_regime, ret_var,
                       confidence=CONFIDENCE, position=POSITION_SIZE):
    s_min = max(SCALE_MIN, POLY_ORD + 2)
    s_max = min(SCALE_MAX, len(ret_regime) // 4)
    tau_r = renyi_tau(ret_regime, Q_RENYI,
                      scale_min=s_min, scale_max=s_max)
    V    = renyi_V(Q_RENYI, tau_r)
    V    = V if np.isfinite(V) else 0.0
    mult = 1.0 + V
    bvar = historical_var(ret_var, confidence)
    avar = bvar * mult
    gap  = (avar - bvar) * position
    return V, mult, bvar * 100, avar * 100, gap


# ─────────────────────────────────────────────────────────────────────────────
# DATA  —  BUG FIX HERE
# ─────────────────────────────────────────────────────────────────────────────
_cache = {}

def get_returns(ticker):
    if ticker in _cache:
        return _cache[ticker]
    data  = yf.download(ticker, start=START, end=END, progress=False)
    close = data["Close"].dropna()
    prices = close.values.ravel().astype(float)
    prices = prices[prices > 0]
    rets   = np.diff(np.log(prices))          # length N-1

    # FIX: dates must align with rets, i.e. one entry per return observation.
    # Original code used index[1:] from the dropna frame, but after the
    # positivity filter the frame index no longer matches prices/rets length.
    # We rebuild the positive-price date list then drop the first element so
    # dates[i] corresponds to rets[i] (the return *into* day i).
    pos_mask  = close.values.ravel() > 0
    pos_dates = close.index[pos_mask].tolist()
    dates     = pos_dates[1:]                 # length N-1, same as rets

    assert len(dates) == len(rets), (
        f"{ticker}: dates({len(dates)}) != rets({len(rets)})"
    )
    _cache[ticker] = (prices, rets, dates)
    return _cache[ticker]


def portfolio_returns(tickers):
    """Equal-weight portfolio log-returns on the intersection of trading dates."""
    all_rets = []
    for t in tickers:
        _, r, d = get_returns(t)
        all_rets.append((d, r))

    # intersection of date sets
    common = sorted(
        set(all_rets[0][0]).intersection(*[set(d) for d, _ in all_rets[1:]])
    )
    if len(common) < 200:
        return np.array([]), []

    port_r = np.zeros(len(common))
    for dates_i, rets_i in all_rets:
        idx_map = {d: i for i, d in enumerate(dates_i)}
        for k, cd in enumerate(common):
            port_r[k] += rets_i[idx_map[cd]]   # idx now always < len(rets_i)

    return port_r / len(tickers), common


def rolling_renyi_var(returns):
    N = len(returns)
    start = max(REGIME_WINDOW, VAR_WINDOW)
    Vs, mults, bvars, avars, gaps = [], [], [], [], []
    for i in range(start, N):
        V, m, bv, av, g = renyi_adjusted_var(
            returns[i - REGIME_WINDOW:i],
            returns[i - VAR_WINDOW:i]
        )
        Vs.append(V); mults.append(m)
        bvars.append(bv); avars.append(av); gaps.append(g)
    return (np.array(Vs), np.array(mults),
            np.array(bvars), np.array(avars), np.array(gaps))


# ─────────────────────────────────────────────────────────────────────────────
# 6-PANEL FIGURE PER PORTFOLIO
# ─────────────────────────────────────────────────────────────────────────────
def plot_portfolio(pname, tickers, port_color, scores):
    print(f"\n  {'─'*56}")
    print(f"  Portfolio : {pname}  ({', '.join(tickers)})")
    print(f"  {'─'*56}")

    hq_dict, tau_dict, alpha_dict, Da_dict = {}, {}, {}, {}
    da_vals, hslope_vals, minr_vals = [], [], []

    for ticker in tickers:
        print(f"    MF-DFA  {ticker} …", end=" ", flush=True)
        _, rets, _ = get_returns(ticker)
        s_min = max(SCALE_MIN, POLY_ORD + 2)
        s_max = min(SCALE_MAX, len(rets) // 4)
        _, _, hq         = mfdfa(rets, Q_VALS, scale_min=s_min, scale_max=s_max)
        tau_q, alpha, Da = multifractal_spectrum(Q_VALS, hq)
        da  = delta_alpha(alpha, Da)
        hsl = hq_slope(Q_VALS, hq)
        print(f"Δα={da:.4f}  |dH/dq|={hsl:.5f}")
        hq_dict[ticker]    = hq
        tau_dict[ticker]   = tau_q
        alpha_dict[ticker] = alpha
        Da_dict[ticker]    = Da
        da_vals.append(da)
        hslope_vals.append(hsl)
        minr_vals.append(MIN_R.get(ticker, np.nan))

    print(f"    Rolling Rényi VaR …", flush=True)
    port_r, common_dates = portfolio_returns(tickers)
    Vs, mults, bvars, avars, gaps = rolling_renyi_var(port_r)

    n_common    = len(port_r)
    cumret_port = np.exp(np.cumsum(port_r)) - 1.0

    mean_da     = float(np.nanmean(da_vals))
    mean_hslope = float(np.nanmean(hslope_vals))
    mean_V      = float(np.nanmean(Vs))
    mean_gap    = float(np.nanmean(gaps)) / 1000.0
    mean_minr   = float(np.nanmean(minr_vals))
    scores[pname] = {
        "Δα":         mean_da,
        "H(q) slope": mean_hslope,
        "V̄":          mean_V,
        "VaR gap $K": mean_gap,
        "min r":      mean_minr,
    }
    print(f"    Summary → Δα={mean_da:.4f}  H-slope={mean_hslope:.5f}  "
          f"V̄={mean_V:.4f}  gap=${mean_gap:.1f}K  min-r={mean_minr:.4f}")

    # ── FIGURE ────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(18, 16))
    gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.50, wspace=0.30)
    axes = [fig.add_subplot(gs[r, c]) for r in range(3) for c in range(2)]

    fig.suptitle(
        f"Multifractal Analysis  ·  {pname} Portfolio\n"
        f"Δα={mean_da:.4f}  ·  ⟨V⟩={mean_V:.4f}  ·  "
        f"⟨|dH/dq|⟩={mean_hslope:.5f}  ·  ⟨min r⟩={mean_minr:.4f}",
        fontsize=13, fontweight="bold"
    )

    # (a) Cumulative return ────────────────────────────────────────────────
    ax = axes[0]
    for ti, ticker in enumerate(tickers):
        _, rets_i, dates_i = get_returns(ticker)
        idx_map = {d: i for i, d in enumerate(dates_i)}
        aligned = np.array([rets_i[idx_map[cd]]
                             for cd in common_dates if cd in idx_map])
        if len(aligned) == 0:
            continue
        cr = np.exp(np.cumsum(aligned)) - 1.0
        ax.plot(cr * 100, color=TICKER_PALETTE[ti], lw=0.9,
                alpha=0.55, label=ticker)
    ax.plot(cumret_port * 100, color=port_color, lw=2.4,
            label="Portfolio (EW)", zorder=5)
    ax.axhline(0, color="#555", lw=0.6, ls="--")
    ax.set_ylabel("Cumulative return (%)", fontsize=9)
    ax.set_title("(a) Cumulative return — equal-weighted",
                 fontsize=10, fontweight="semibold")
    ax.legend(fontsize=7.5, ncol=3, loc="upper left", framealpha=0.85)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda v, _: f"{v:.0f}%"))
    ax.tick_params(axis="x", labelbottom=False)

    # (b) H(q) — full spectrum ────────────────────────────────────────────
    ax = axes[1]
    for ti, ticker in enumerate(tickers):
        hq    = hq_dict[ticker]
        valid = np.isfinite(hq)
        hsl_i = hq_slope(Q_VALS, hq)
        ax.plot(Q_VALS[valid], hq[valid],
                color=TICKER_PALETTE[ti], lw=1.7,
                label=f"{ticker}  slope={hsl_i:.4f}")
    ax.axhline(0.5, color="#999", lw=0.8, ls="--", alpha=0.6,
               label="H=0.5 (random walk)")
    ax.axhline(1.0, color="#66a", lw=0.7, ls=":",  alpha=0.5,
               label="H=1.0")
    ax.axvline(0,   color="#ccc", lw=0.5)
    ax.set_xlabel("q", fontsize=9)
    ax.set_ylabel("H(q)", fontsize=9)
    ax.set_title(
        "(b) H(q) — full generalised Hurst spectrum  [q ∈ (−30, +30), step=5]\n"
        "Slope of curve = multifractality  ·  level at q=2 = classical H",
        fontsize=10, fontweight="semibold"
    )
    ax.legend(fontsize=7.5, ncol=1, loc="upper right", framealpha=0.85)

    # (c) τ(q) ────────────────────────────────────────────────────────────
    ax = axes[2]
    tau_mono = Q_VALS * 0.5 - 1.0
    ax.plot(Q_VALS, tau_mono, color="#bbb", lw=1.2, ls="--",
            label="Monofractal ref (H=0.5)", zorder=1)
    for ti, ticker in enumerate(tickers):
        tau_q = tau_dict[ticker]
        valid  = np.isfinite(tau_q)
        ax.plot(Q_VALS[valid], tau_q[valid],
                color=TICKER_PALETTE[ti], lw=1.7, label=ticker, zorder=2)
    ax.axhline(0, color="#555", lw=0.5, ls=":")
    ax.set_xlabel("q", fontsize=9)
    ax.set_ylabel("τ(q)", fontsize=9)
    ax.set_title(
        "(c) τ(q) = q·H(q) − 1  — mass scaling exponent\n"
        "Deviation from dashed line = degree of multifractality",
        fontsize=10, fontweight="semibold"
    )
    ax.legend(fontsize=7.5, ncol=2, loc="upper left", framealpha=0.85)

    # (d) f(α) ────────────────────────────────────────────────────────────
    # Find q=0 index once (same for every ticker — it's the grid apex)
    q0_idx = np.argmin(np.abs(Q_VALS))   # index of q closest to 0

    ax = axes[3]
    all_alpha_vals, all_Da_vals = [], []   # collect for axis auto-clip

    for ti, ticker in enumerate(tickers):
        alpha_i = alpha_dict[ticker]
        Da_i    = Da_dict[ticker]
        valid   = np.isfinite(alpha_i) & np.isfinite(Da_i)
        da_i    = delta_alpha(alpha_i, Da_i)

        ax.plot(alpha_i[valid], Da_i[valid],
                color=TICKER_PALETTE[ti], lw=1.7,
                label=f"{ticker}  Δα={da_i:.3f}")

        # Mark the q=0 apex (f(α) peak = box-counting dimension ≈ 1)
        if valid[q0_idx]:
            ax.plot(alpha_i[q0_idx], Da_i[q0_idx], "o",
                    color=TICKER_PALETTE[ti], markersize=5,
                    markeredgecolor="white", markeredgewidth=0.7, zorder=5)

        all_alpha_vals.extend(alpha_i[valid].tolist())
        all_Da_vals.extend(Da_i[valid].tolist())

    ax.axhline(1.0, color="#999", lw=0.7, ls="--", alpha=0.5,
               label="f(α)=1 reference")

    # Auto-clip axes to the meaningful parabola region (exclude wild tails)
    if all_alpha_vals:
        a_lo = np.percentile(all_alpha_vals,  2)
        a_hi = np.percentile(all_alpha_vals, 98)
        d_lo = max(0, np.percentile(all_Da_vals,  2) - 0.05)
        d_hi = np.percentile(all_Da_vals, 98) + 0.05
        ax.set_xlim(a_lo - 0.02, a_hi + 0.02)
        ax.set_ylim(d_lo, min(d_hi, 1.4))

    ax.set_xlabel("α  (singularity strength)", fontsize=9)
    ax.set_ylabel("f(α)  (fractal dimension)", fontsize=9)
    ax.set_title(
        "(d) f(α) singularity spectrum  —  dots mark q=0 apex\n"
        "Width Δα = multifractal heterogeneity  ·  apex ≈ box-counting dim",
        fontsize=10, fontweight="semibold"
    )
    ax.legend(fontsize=7.5, ncol=1, loc="upper right", framealpha=0.85)

    # (e) Rolling Rényi V(t) ───────────────────────────────────────────────
    ax = axes[4]
    xs_r = np.arange(len(Vs))
    ax.fill_between(xs_r, 0, Vs, color=port_color, alpha=0.30, zorder=1)
    ax.plot(xs_r, Vs, color=port_color, lw=1.5,
            label="V(t) = std(τ_R curvature residuals)", zorder=2)
    ax.axhline(mean_V, color=port_color, lw=1.1, ls="--", alpha=0.85,
               label=f"⟨V⟩ = {mean_V:.4f}", zorder=3)
    ax.set_ylabel("V(t)", fontsize=9)
    ax.set_title(
        "(e) Rolling Rényi volatility index V(t)\n"
        "VaR multiplier = 1 + V(t)  ·  no constants",
        fontsize=10, fontweight="semibold"
    )
    ax.legend(fontsize=8, loc="upper right", framealpha=0.85)
    ax.tick_params(axis="x", labelbottom=False)

    # (f) Rényi-adjusted VaR ──────────────────────────────────────────────
    ax = axes[5]
    xs_r = np.arange(len(bvars))
    ax.plot(xs_r, bvars, color="#888780", lw=1.1, ls="--",
            label="Standard VaR (99%)")
    ax.plot(xs_r, avars, color="#A32D2D", lw=1.8,
            label="Rényi-adjusted VaR")
    ax.fill_between(xs_r, bvars, avars,
                    color="#F09595", alpha=0.45, label="Hidden risk gap")
    ax.set_ylabel("VaR (%)", fontsize=9)
    ax.set_title(
        f"(f) VaR comparison  ·  ⟨gap⟩ = ${mean_gap:.1f}K "
        f"per ${POSITION_SIZE:,}",
        fontsize=10, fontweight="semibold"
    )
    ax.legend(fontsize=8, loc="upper right", framealpha=0.85)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda v, _: f"{v:.3f}%"))

    for ax in axes:
        ax.tick_params(direction="in", which="both", top=True, right=True)

    fname = (f"{OUT_DIR}/"
             f"{pname.replace(' ','_').replace('-','_')}_6panel.png")
    fig.savefig(fname, dpi=130, bbox_inches="tight")
    plt.close(fig)
    print(f"    Saved → {fname}")


# ─────────────────────────────────────────────────────────────────────────────
# CROSS-PORTFOLIO SCORECARD
# ─────────────────────────────────────────────────────────────────────────────
def _rank_score(values, lower_is_better=True):
    arr   = np.array(values, dtype=float)
    valid = np.isfinite(arr)
    out   = np.full(len(arr), 0.5)
    if valid.sum() < 2:
        return out
    mn, mx = arr[valid].min(), arr[valid].max()
    if mx == mn:
        return out
    normed = (arr - mn) / (mx - mn)
    out[valid] = (1 - normed[valid]) if lower_is_better else normed[valid]
    return out


def plot_scorecard(scores):
    pnames  = list(scores.keys())
    metrics = ["Δα", "H(q) slope", "V̄", "VaR gap $K", "min r"]
    better  = [True,  True,         True, True,          False]

    raw = np.array([[scores[p][m] for m in metrics] for p in pnames])
    ranked = np.stack([_rank_score(raw[:, mi], lb)
                       for mi, lb in enumerate(better)], axis=1)
    composite = ranked.mean(axis=1)
    best_idx  = int(np.argmax(composite))

    print(f"\n{'═'*72}")
    print(f"  CROSS-PORTFOLIO MULTIFRACTAL RISK SCORECARD")
    print(f"{'═'*72}")
    hdr = (f"  {'Portfolio':<22}" +
           "".join(f"{m:>13}" for m in metrics) +
           f"  {'Score':>7}")
    print(hdr)
    print("  " + "─" * (len(hdr) - 2))
    for pi, pname in enumerate(pnames):
        row = f"  {pname:<22}"
        for mi, m in enumerate(metrics):
            row += f"{scores[pname][m]:>13.4f}"
        row += f"  {composite[pi]:>7.4f}"
        if pi == best_idx:
            row += "  ← BEST (lowest risk complexity)"
        print(row)
    print(f"\n  Winner: {pnames[best_idx]}")
    print(f"  Scoring derived from full H(q) curves.  H(q=2) not used standalone.")

    fig, axes = plt.subplots(1, 2, figsize=(17, 6))
    fig.suptitle(
        "Cross-Portfolio Multifractal Risk Scorecard\n"
        "Scores derived from full H(q) spectra  —  "
        "H(q=2) not used as standalone metric",
        fontsize=13, fontweight="bold"
    )

    ax    = axes[0]
    x     = np.arange(len(pnames))
    w     = 0.14
    cmap  = plt.cm.Set2
    for mi, m in enumerate(metrics):
        ax.bar(x + mi * w, ranked[:, mi], w,
               label=m, color=cmap(mi / len(metrics)), alpha=0.88,
               edgecolor="#333", linewidth=0.5)
    ax.set_xticks(x + w * (len(metrics) - 1) / 2)
    ax.set_xticklabels(pnames, rotation=22, ha="right", fontsize=9)
    ax.set_ylabel("Normalised score  [0 = worst  ·  1 = best]", fontsize=10)
    ax.set_title("Per-metric normalised scores", fontsize=11)
    ax.legend(fontsize=8.5, loc="upper right", framealpha=0.92)
    ax.set_ylim(0, 1.18)
    ax.axhline(0.5, color="#aaa", lw=0.7, ls="--")

    ax     = axes[1]
    colors = [PORTFOLIO_COLORS.get(p, "#999") for p in pnames]
    bars   = ax.bar(pnames, composite, color=colors, alpha=0.88,
                    edgecolor="#333", linewidth=0.7)
    bars[best_idx].set_edgecolor("#FFD700")
    bars[best_idx].set_linewidth(3.5)
    for bar, val in zip(bars, composite):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.007,
                f"{val:.3f}", ha="center", va="bottom",
                fontsize=9.5, fontweight="bold")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Composite score  [1 = safest / most monofractal]",
                  fontsize=10)
    ax.set_title(
        f"Composite risk-complexity score\n"
        f"★  {pnames[best_idx]}  is the most well-behaved portfolio",
        fontsize=11
    )
    ax.set_xticklabels(pnames, rotation=22, ha="right", fontsize=9)

    note = (
        "Metrics (all from full H(q) curves, q ∈ (−30, +30), step=5, q=0 included):\n"
        "  Δα         = f(α) spectrum width             [lower → more monofractal]\n"
        "  H(q) slope = mean |dH/dq| over all q         [lower → less heterogeneous]\n"
        "  V̄          = mean Rényi τ_R curvature index  [lower → VaR more reliable]\n"
        "  VaR gap    = mean hidden $K risk              [lower → safer]\n"
        "  min r      = min Pearson corr F₊q vs F₋q     [higher → symmetric scaling]"
    )
    fig.text(0.5, -0.10, note, ha="center", fontsize=8.5,
             family="monospace", color="#333",
             bbox=dict(boxstyle="round,pad=0.6", fc="#f9f9f9", ec="#ccc"))

    plt.tight_layout()
    fname = f"{OUT_DIR}/scorecard.png"
    fig.savefig(fname, dpi=130, bbox_inches="tight")
    plt.close(fig)
    print(f"\n  Scorecard → {fname}")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    all_tickers = sorted({t for tl in PORTFOLIOS.values() for t in tl})
    print("Pre-downloading data …")
    for t in all_tickers:
        try:
            get_returns(t)
            print(f"  {t} ✓")
        except Exception as e:
            print(f"  {t} FAILED: {e}")

    scores = {}
    for pname, tickers in PORTFOLIOS.items():
        plot_portfolio(pname, tickers,
                       PORTFOLIO_COLORS[pname], scores)

    plot_scorecard(scores)
    print(f"\nAll outputs in:  {OUT_DIR}/")

Pre-downloading data …
  AAPL ✓
  AMZN ✓
  BAC ✓
  BTC-USD ✓
  CL=F ✓
  ETH-USD ✓
  EWJ ✓
  GC=F ✓
  GS ✓
  JNJ ✓
  JPM ✓
  KO ✓
  MA ✓
  META ✓
  NFLX ✓
  NVDA ✓
  PEP ✓
  PG ✓
  SI=F ✓
  TSLA ✓
  V ✓
  WMT ✓
  XOM ✓

  ────────────────────────────────────────────────────────
  Portfolio : Momentum  (NVDA, TSLA, META, AMZN, NFLX)
  ────────────────────────────────────────────────────────
    MF-DFA  NVDA … Δα=0.3539  |dH/dq|=0.00455
    MF-DFA  TSLA … Δα=0.5146  |dH/dq|=0.00685
    MF-DFA  META … Δα=0.4421  |dH/dq|=0.00578
    MF-DFA  AMZN … Δα=0.5391  |dH/dq|=0.00725
    MF-DFA  NFLX … Δα=0.3851  |dH/dq|=0.00496
    Rolling Rényi VaR …


KeyboardInterrupt: 

In [7]:
"""
Portfolio Multifractal Comparison  —  MPT + Diversification edition
=====================================================================
ENHANCED: Added panels for
  (g) F_q(s) log-log fluctuation functions for all q values
  (h) Pearson correlation between F_{+q}(s) and F_{-q}(s) for q=1..30

Portfolio construction  (all from the same broad universe)
-----------------------------------------------------------
Six portfolios, each built from data rather than hand-picked:

  Min-Variance      Markowitz minimum-variance frontier (long-only)
  Max-Sharpe        Tangency portfolio (long-only)
  Risk-Parity       Equal Risk Contribution  (Maillard et al. 2010)
  Max-Diversification  Choueifong (2008)
  Min-Correlation   Subset selection: lowest avg pairwise |corr|
  Max-Decorrelation  Weights minimising w'|C|w

All optimisations: SLSQP, Σw=1, w≥0, w≤W_CAP.
Covariance: Ledoit-Wolf shrinkage for numerical stability.
"""

import numpy as np
import yfinance as yf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
from scipy.signal   import savgol_filter
from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf
import os, warnings, itertools

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# UNIVERSE  —  broad cross-asset pool
# ─────────────────────────────────────────────────────────────────────────────
UNIVERSE = [
    "AAPL", "MSFT", "NVDA", "AMZN", "META", "TSLA", "NFLX", "AMD",
    "JPM", "GS", "BAC", "V", "MA",
    "JNJ", "PG", "KO", "WMT", "PEP",
    "XOM", "CVX", "CAT", "HON",
    "GC=F", "SI=F", "CL=F", "BTC-USD", "ETH-USD",
    "EWJ", "EWZ", "FXI", "EWG",
]

N_PORT   = 5
RF_DAILY = 0.045/252
W_CAP    = 0.40

START    = "2015-01-01"
END      = "2026-04-11"

# ─────────────────────────────────────────────────────────────────────────────
# MF-DFA PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────
Q_VALS  = np.arange(-30, 31, 5, dtype=float)   # includes q=0
Q_SYM   = np.arange(1, 31, dtype=float)        # for symmetric F corr: q=1..30

Q_RENYI = np.concatenate([
    np.linspace(-20, -0.5, 20),
    np.linspace( 0.5,  20, 20)
])

POLY_ORD      = 1
OVERLAP       = 0.75
N_SCALES      = 30
SCALE_MIN     = 20
SCALE_MAX     = 800
REGIME_WINDOW = 120
VAR_WINDOW    = 60
CONFIDENCE    = 0.99
POSITION_SIZE = 100_000

OUT_DIR = "portfolio_mf_plots_enhanced"
os.makedirs(OUT_DIR, exist_ok=True)

MIN_R = {
    "AAPL": 0.9700, "AMZN": 0.9744, "NVDA": 0.9706, "META": 0.9287,
    "TSLA": 0.9324, "NFLX": 0.9841, "MSFT": 0.9650, "AMD":  0.9410,
    "JPM":  0.9842, "BAC":  0.9786, "GS":   0.9468,
    "V":    0.8369, "MA":   0.7516,
    "JNJ":  0.3982, "PG":   0.8195, "KO":   0.2383, "PEP": -0.1332,
    "WMT":  0.9644, "XOM":  0.9296, "CVX":  0.9210,
    "CAT":  0.9310, "HON":  0.9180,
    "GC=F": 0.9118, "SI=F": 0.8545, "CL=F": 0.7178,
    "BTC-USD": 0.9769, "ETH-USD": 0.9670,
    "EWJ":  0.9581, "EWZ":  0.8920, "FXI":  0.8760, "EWG":  0.9340,
}

PORTFOLIO_COLORS = {
    "Min-Variance":        "#58a6ff",
    "Max-Sharpe":          "#f85149",
    "Risk-Parity":         "#3fb950",
    "Max-Diversification": "#e3b341",
    "Min-Correlation":     "#bc8cff",
    "Max-Decorrelation":   "#ff7f0e",
}

TICKER_PALETTE = [
    "#e63946", "#2a9d8f", "#e9c46a", "#264653", "#a8dadc",
    "#457b9d", "#f4a261", "#6a4c93", "#80b918", "#f72585",
]

# Color ramp for F_q(s) lines — one per q value in Q_VALS
FQ_CMAP = plt.cm.coolwarm


# ─────────────────────────────────────────────────────────────────────────────
# DATA LAYER
# ─────────────────────────────────────────────────────────────────────────────
_cache = {}

def get_returns(ticker):
    """Return (prices, log_returns, dates) with dates aligned to returns."""
    if ticker in _cache:
        return _cache[ticker]
    data   = yf.download(ticker, start=START, end=END, progress=False)
    close  = data["Close"].dropna()
    prices = close.values.ravel().astype(float)
    prices = prices[prices > 0]
    rets   = np.diff(np.log(prices))

    pos_mask  = close.values.ravel() > 0
    pos_dates = close.index[pos_mask].tolist()
    dates     = pos_dates[1:]

    assert len(dates) == len(rets), \
        f"{ticker}: dates({len(dates)}) != rets({len(rets)})"
    _cache[ticker] = (prices, rets, dates)
    return _cache[ticker]


def build_return_matrix(tickers):
    all_data = {}
    for t in tickers:
        try:
            _, r, d = get_returns(t)
            all_data[t] = {date: ret for date, ret in zip(d, r)}
        except Exception:
            pass

    if not all_data:
        return None, [], []

    common = sorted(
        set.intersection(*[set(v.keys()) for v in all_data.values()])
    )
    if len(common) < 200:
        return None, [], []

    valid_tickers = list(all_data.keys())
    mat = np.array(
        [[all_data[t][d] for d in common] for t in valid_tickers]
    ).T

    return mat, common, valid_tickers


# ─────────────────────────────────────────────────────────────────────────────
# PORTFOLIO OPTIMISATION
# ─────────────────────────────────────────────────────────────────────────────

def _opt(fun, N, w0=None):
    if w0 is None:
        w0 = np.full(N, 1.0 / N)
    constraints = {"type": "eq", "fun": lambda w: w.sum() - 1.0}
    bounds      = [(0.0, W_CAP)] * N
    res = minimize(fun, w0, method="SLSQP",
                   bounds=bounds, constraints=constraints,
                   options={"maxiter": 2000, "ftol": 1e-14})
    if res.success:
        w = np.clip(res.x, 0, None)
        return w / w.sum()
    return w0


def min_variance_weights(cov):
    N = cov.shape[0]
    return _opt(lambda w: w @ cov @ w, N)


def max_sharpe_weights(mu, cov, rf=RF_DAILY):
    N = cov.shape[0]
    def neg_sharpe(w):
        ret = w @ mu
        vol = np.sqrt(w @ cov @ w)
        return -(ret - rf) / vol if vol > 1e-10 else 0.0
    return _opt(neg_sharpe, N)


def risk_parity_weights(cov, tol=1e-12, max_iter=1000):
    N = cov.shape[0]
    w = np.full(N, 1.0 / N)
    for _ in range(max_iter):
        sigma  = np.sqrt(w @ cov @ w)
        mrc    = cov @ w / sigma
        rc     = w * mrc
        target = rc.mean()
        w_new  = w * (target / np.clip(rc, 1e-12, None))
        w_new /= w_new.sum()
        if np.max(np.abs(w_new - w)) < tol:
            return w_new
        w = w_new
    return w / w.sum()


def max_diversification_weights(cov):
    N    = cov.shape[0]
    vols = np.sqrt(np.diag(cov))
    def neg_dr(w):
        port_vol = np.sqrt(w @ cov @ w)
        return -(w @ vols) / port_vol if port_vol > 1e-10 else 0.0
    return _opt(neg_dr, N)


def min_correlation_selection(corr, tickers, n=N_PORT):
    N = len(tickers)
    best_corr = np.inf
    best_idx  = list(range(n))

    if N <= 20:
        for combo in itertools.combinations(range(N), n):
            idx  = list(combo)
            sub  = corr[np.ix_(idx, idx)]
            mask = np.triu(np.ones_like(sub, dtype=bool), k=1)
            avg  = np.abs(sub[mask]).mean()
            if avg < best_corr:
                best_corr = avg
                best_idx  = idx
    else:
        min_val, pi, pj = np.inf, 0, 1
        for i in range(N):
            for j in range(i+1, N):
                if abs(corr[i, j]) < min_val:
                    min_val, pi, pj = abs(corr[i, j]), i, j
        used = {pi, pj}
        while len(used) < n:
            best_gain, best_k = np.inf, -1
            for k in range(N):
                if k in used:
                    continue
                avg = np.mean([abs(corr[k, j]) for j in used])
                if avg < best_gain:
                    best_gain, best_k = avg, k
            used.add(best_k)
        best_idx = sorted(used)

    selected = [tickers[i] for i in best_idx]
    weights  = np.full(n, 1.0 / n)
    return selected, weights


def max_decorrelation_weights(corr):
    N        = corr.shape[0]
    abs_corr = np.abs(corr)
    return _opt(lambda w: w @ abs_corr @ w, N)


def _top_n(tickers, weights, n, label=""):
    idx   = np.argsort(weights)[::-1][:n]
    sel_t = [tickers[i] for i in idx]
    sel_w = weights[idx]
    sel_w = sel_w / sel_w.sum()
    print(f"  {label:<24} → {sel_t}")
    print(f"  {'':24}   weights={np.round(sel_w, 3)}")
    return {"tickers": sel_t, "weights": sel_w}


# ─────────────────────────────────────────────────────────────────────────────
# BUILD ALL PORTFOLIOS
# ─────────────────────────────────────────────────────────────────────────────
def build_portfolios(universe):
    print("Building universe return matrix …")
    mat, common, tickers = build_return_matrix(universe)
    if mat is None or len(tickers) < N_PORT:
        raise RuntimeError("Insufficient universe data.")

    T, N = mat.shape
    print(f"  {N} assets  ·  {T} common trading days")

    lw   = LedoitWolf().fit(mat)
    cov  = lw.covariance_
    mu   = mat.mean(axis=0)
    vols = np.sqrt(np.diag(cov))
    corr = cov / np.outer(vols, vols)
    np.fill_diagonal(corr, 1.0)

    print("\nRunning optimisers …")
    portfolios = {}

    portfolios["Min-Variance"] = _top_n(
        tickers, min_variance_weights(cov), N_PORT, "Min-Variance")

    portfolios["Max-Sharpe"] = _top_n(
        tickers, max_sharpe_weights(mu, cov), N_PORT, "Max-Sharpe")

    portfolios["Risk-Parity"] = _top_n(
        tickers, risk_parity_weights(cov), N_PORT, "Risk-Parity")

    portfolios["Max-Diversification"] = _top_n(
        tickers, max_diversification_weights(cov), N_PORT,
        "Max-Diversification")

    sel_t, sel_w = min_correlation_selection(corr, tickers, N_PORT)
    portfolios["Min-Correlation"] = {"tickers": sel_t, "weights": sel_w}
    print(f"  {'Min-Correlation':<24} → {sel_t}")
    print(f"  {'':24}   weights={np.round(sel_w,3)}  (equal-weight)")

    portfolios["Max-Decorrelation"] = _top_n(
        tickers, max_decorrelation_weights(corr), N_PORT,
        "Max-Decorrelation")

    return portfolios


# ─────────────────────────────────────────────────────────────────────────────
# WEIGHTED PORTFOLIO RETURNS
# ─────────────────────────────────────────────────────────────────────────────
def weighted_portfolio_returns(tickers, weights):
    weights = np.asarray(weights, dtype=float)
    mat, common, valid = build_return_matrix(tickers)
    if mat is None:
        return np.array([]), []

    w_map = {t: weights[tickers.index(t)] for t in valid if t in tickers}
    w     = np.array([w_map.get(t, 0.0) for t in valid])
    w    /= w.sum()

    return mat @ w, common


def rolling_renyi_var(returns):
    N     = len(returns)
    start = max(REGIME_WINDOW, VAR_WINDOW)
    Vs, mults, bvars, avars, gaps = [], [], [], [], []
    for i in range(start, N):
        V, m, bv, av, g = renyi_adjusted_var(
            returns[i - REGIME_WINDOW:i],
            returns[i - VAR_WINDOW:i]
        )
        Vs.append(V); mults.append(m)
        bvars.append(bv); avars.append(av); gaps.append(g)
    return (np.array(Vs), np.array(mults),
            np.array(bvars), np.array(avars), np.array(gaps))


# ─────────────────────────────────────────────────────────────────────────────
# MF-DFA CORE
# ─────────────────────────────────────────────────────────────────────────────
def integrate(x):
    return np.cumsum(x - x.mean())


def _overlap_starts(N, s, overlap):
    step = max(1, int(s * (1.0 - overlap)))
    return np.arange(0, N - s + 1, step)


def _segment_rms_list(x, scale_min, scale_max, n_scales,
                      poly_ord=POLY_ORD, overlap=OVERLAP):
    y         = integrate(x)
    N         = len(y)
    scale_max = min(scale_max, N // 4)
    scale_min = max(scale_min, poly_ord + 2)
    if scale_max < scale_min:
        return np.array([]), []
    scales = np.unique(
        np.logspace(np.log10(scale_min), np.log10(scale_max),
                    n_scales).astype(int)
    )
    scales  = scales[scales >= scale_min]
    rms_per = []
    for s in scales:
        starts = _overlap_starts(N, s, overlap)
        if len(starts) < 4:
            rms_per.append(np.array([])); continue
        t, rms = np.arange(s), []
        for st in starts:
            seg  = y[st:st+s]
            coef = np.polyfit(t, seg, poly_ord)
            r    = np.sqrt(np.mean((seg - np.polyval(coef, t))**2))
            if r > 0:
                rms.append(r)
        rms_per.append(np.asarray(rms))
    return scales, rms_per


def mfdfa(x, q_vals, scale_min=SCALE_MIN, scale_max=SCALE_MAX,
          n_scales=N_SCALES, poly_ord=POLY_ORD, overlap=OVERLAP):
    scales, rms_per = _segment_rms_list(x, scale_min, scale_max,
                                         n_scales, poly_ord, overlap)
    if len(scales) == 0:
        return (np.array([]),
                np.full((len(q_vals), 0), np.nan),
                np.full(len(q_vals), np.nan))

    Fqs = np.full((len(q_vals), len(scales)), np.nan)
    for i, rms in enumerate(rms_per):
        if len(rms) == 0:
            continue
        for j, q in enumerate(q_vals):
            if q == 0:
                Fqs[j, i] = np.exp(0.5 * np.mean(np.log(rms**2)))
            else:
                Fqs[j, i] = (np.mean(rms**q))**(1.0/q)

    hq = np.full(len(q_vals), np.nan)
    for j in range(len(q_vals)):
        row  = Fqs[j]
        mask = np.isfinite(row) & (row > 0)
        if mask.sum() >= 4:
            hq[j] = np.polyfit(np.log(scales[mask]),
                                np.log(row[mask]), 1)[0]
    return scales, Fqs, hq


def mfdfa_symmetric(x, q_vals_pos, scale_min=SCALE_MIN, scale_max=SCALE_MAX,
                    n_scales=N_SCALES, poly_ord=POLY_ORD, overlap=OVERLAP):
    """
    Compute F_{+q}(s) and F_{-q}(s) separately for q > 0.
    Returns scales, Fqs_pos, Fqs_neg  — shape (len(q_vals_pos), len(scales))
    """
    scales, rms_per = _segment_rms_list(x, scale_min, scale_max,
                                         n_scales, poly_ord, overlap)
    if len(scales) == 0:
        empty = np.full((len(q_vals_pos), 0), np.nan)
        return np.array([]), empty, empty

    Fqs_pos = np.full((len(q_vals_pos), len(scales)), np.nan)
    Fqs_neg = np.full((len(q_vals_pos), len(scales)), np.nan)

    for i, rms in enumerate(rms_per):
        if len(rms) == 0:
            continue
        for j, q in enumerate(q_vals_pos):
            Fqs_pos[j, i] = (np.mean(rms ** q))  ** (1.0 /  q)
            Fqs_neg[j, i] = (np.mean(rms ** -q)) ** (1.0 / -q)

    return scales, Fqs_pos, Fqs_neg


def pearson_sym_corr(Fqs_pos_row, Fqs_neg_row):
    """Pearson r between log(F_{+q}) and log(F_{-q}) across scales."""
    mask = (np.isfinite(Fqs_pos_row) & np.isfinite(Fqs_neg_row) &
            (Fqs_pos_row > 0) & (Fqs_neg_row > 0))
    if mask.sum() < 3:
        return np.nan
    a = np.log(Fqs_pos_row[mask])
    b = np.log(Fqs_neg_row[mask])
    denom = a.std() * b.std()
    if denom < 1e-12:
        return np.nan
    return float(np.mean((a - a.mean()) * (b - b.mean())) / denom)


def multifractal_spectrum(q_vals, hq):
    tau_q = q_vals * hq - 1.0
    n  = len(tau_q)
    wl = min(5, n if n % 2 == 1 else n - 1)
    wl = max(wl, 3)
    valid_mask = np.isfinite(tau_q)
    tau_s = tau_q.copy()
    if valid_mask.sum() >= wl:
        tau_s[valid_mask] = savgol_filter(tau_q[valid_mask], wl, 2)
    alpha = np.gradient(tau_s, q_vals)
    D_a   = q_vals * alpha - tau_s
    return tau_q, alpha, D_a


def delta_alpha(alpha, D_a):
    valid = np.isfinite(alpha) & np.isfinite(D_a)
    if valid.sum() < 2:
        return np.nan
    return float(alpha[valid].max() - alpha[valid].min())


def hq_slope(q_vals, hq):
    valid = np.isfinite(hq)
    if valid.sum() < 4:
        return np.nan
    return float(np.mean(np.abs(np.gradient(hq[valid], q_vals[valid]))))


# ─────────────────────────────────────────────────────────────────────────────
# RÉNYI PARTITION FUNCTION
# ─────────────────────────────────────────────────────────────────────────────
def _log_chi_q(rms, q):
    if len(rms) < 3:
        return np.nan
    p = rms**2 / (rms**2).sum()
    if q == 1:
        p_s = p[p > 0]
        return float(-np.sum(p_s * np.log(p_s)))
    val = np.sum(p**q)
    return float(np.log(val)) if val > 0 else np.nan


def renyi_tau(x, q_vals, scale_min=SCALE_MIN, scale_max=SCALE_MAX,
              n_scales=N_SCALES, poly_ord=POLY_ORD, overlap=OVERLAP):
    scales, rms_per = _segment_rms_list(x, scale_min, scale_max,
                                         n_scales, poly_ord, overlap)
    if len(scales) == 0:
        return np.full(len(q_vals), np.nan)
    tau_r = np.full(len(q_vals), np.nan)
    for j, q in enumerate(q_vals):
        lchi = np.array([_log_chi_q(r, q) for r in rms_per])
        mask = np.isfinite(lchi) & \
               (np.array([len(r) for r in rms_per]) >= 3)
        if mask.sum() >= 4:
            tau_r[j] = np.polyfit(np.log(scales[mask]),
                                   lchi[mask], 1)[0]
    return tau_r


def renyi_V(q_vals, tau_r):
    valid = np.isfinite(tau_r)
    if valid.sum() < 6:
        return np.nan
    q_v, t_v = q_vals[valid], tau_r[valid]
    resid = t_v - np.polyval(np.polyfit(q_v, t_v, 1), q_v)
    return float(np.std(resid))


# ─────────────────────────────────────────────────────────────────────────────
# VaR
# ─────────────────────────────────────────────────────────────────────────────
def historical_var(returns, confidence=CONFIDENCE):
    return -np.percentile(returns, (1.0 - confidence) * 100.0)


def renyi_adjusted_var(ret_regime, ret_var,
                       confidence=CONFIDENCE, position=POSITION_SIZE):
    s_min = max(SCALE_MIN, POLY_ORD + 2)
    s_max = min(SCALE_MAX, len(ret_regime) // 4)
    tau_r = renyi_tau(ret_regime, Q_RENYI,
                      scale_min=s_min, scale_max=s_max)
    V    = renyi_V(Q_RENYI, tau_r)
    V    = V if np.isfinite(V) else 0.0
    mult = 1.0 + V
    bvar = historical_var(ret_var, confidence)
    avar = bvar * mult
    gap  = (avar - bvar) * position
    return V, mult, bvar * 100, avar * 100, gap


# ─────────────────────────────────────────────────────────────────────────────
# 8-PANEL FIGURE PER PORTFOLIO
# ─────────────────────────────────────────────────────────────────────────────
def plot_portfolio(pname, tickers, weights, port_color, scores):
    weights = np.asarray(weights, dtype=float)
    print(f"\n  {'─'*60}")
    print(f"  Portfolio : {pname}")
    for t, w in zip(tickers, weights):
        print(f"    {t:<12}  {w*100:5.1f}%")

    hq_dict, tau_dict, alpha_dict, Da_dict = {}, {}, {}, {}
    fqs_dict       = {}   # all-q fluctuation functions
    scales_dict    = {}   # scales per ticker
    sym_corr_dict  = {}   # symmetric F corr per ticker
    da_vals, hslope_vals, minr_vals = [], [], []

    for ticker in tickers:
        print(f"    MF-DFA  {ticker} …", end=" ", flush=True)
        _, rets, _ = get_returns(ticker)
        s_min = max(SCALE_MIN, POLY_ORD + 2)
        s_max = min(SCALE_MAX, len(rets) // 4)

        # Full q-range DFA (for H(q), tau, spectrum)
        scales_all, Fqs_all, hq = mfdfa(rets, Q_VALS,
                                         scale_min=s_min, scale_max=s_max)
        tau_q, alpha, Da = multifractal_spectrum(Q_VALS, hq)
        da  = delta_alpha(alpha, Da)
        hsl = hq_slope(Q_VALS, hq)
        print(f"Δα={da:.4f}  |dH/dq|={hsl:.5f}")

        hq_dict[ticker]    = hq
        tau_dict[ticker]   = tau_q
        alpha_dict[ticker] = alpha
        Da_dict[ticker]    = Da
        fqs_dict[ticker]   = Fqs_all
        scales_dict[ticker] = scales_all
        da_vals.append(da)
        hslope_vals.append(hsl)
        minr_vals.append(MIN_R.get(ticker, np.nan))

        # Symmetric F_{+q} / F_{-q} for q=1..30
        _, Fqs_pos, Fqs_neg = mfdfa_symmetric(rets, Q_SYM,
                                               scale_min=s_min, scale_max=s_max)
        corr_vals = np.array([
            pearson_sym_corr(Fqs_pos[j], Fqs_neg[j])
            for j in range(len(Q_SYM))
        ])
        sym_corr_dict[ticker] = corr_vals

    print(f"    Rolling Rényi VaR …", flush=True)
    port_r, common_dates = weighted_portfolio_returns(tickers, weights)
    if len(port_r) == 0:
        print(f"    Skipping {pname}: no common dates.")
        return

    Vs, mults, bvars, avars, gaps = rolling_renyi_var(port_r)
    cumret_port = np.exp(np.cumsum(port_r)) - 1.0

    mean_da     = float(np.nanmean(da_vals))
    mean_hslope = float(np.nanmean(hslope_vals))
    mean_V      = float(np.nanmean(Vs))
    mean_gap    = float(np.nanmean(gaps)) / 1000.0
    mean_minr   = float(np.nanmean(minr_vals))
    scores[pname] = {
        "Δα":         mean_da,
        "H(q) slope": mean_hslope,
        "V̄":          mean_V,
        "VaR gap $K": mean_gap,
        "min r":      mean_minr,
    }
    print(f"    → Δα={mean_da:.4f}  H-slope={mean_hslope:.5f}  "
          f"V̄={mean_V:.4f}  gap=${mean_gap:.1f}K  min-r={mean_minr:.4f}")

    # ── FIGURE  8 panels (4 rows × 2 cols) ────────────────────────────────
    fig = plt.figure(figsize=(18, 22))
    gs  = gridspec.GridSpec(4, 2, figure=fig, hspace=0.54, wspace=0.32)
    axes = [fig.add_subplot(gs[r, c]) for r in range(4) for c in range(2)]

    w_str = "  ".join(f"{t} {w*100:.0f}%" for t, w in zip(tickers, weights))
    fig.suptitle(
        f"Multifractal Analysis  ·  {pname} Portfolio\n"
        f"{w_str}\n"
        f"Δα={mean_da:.4f}  ·  ⟨V⟩={mean_V:.4f}  ·  "
        f"⟨|dH/dq|⟩={mean_hslope:.5f}  ·  ⟨min r⟩={mean_minr:.4f}",
        fontsize=11, fontweight="bold"
    )

    # (a) Cumulative return ────────────────────────────────────────────────
    ax = axes[0]
    for ti, ticker in enumerate(tickers):
        _, rets_i, dates_i = get_returns(ticker)
        idx_map = {d: i for i, d in enumerate(dates_i)}
        aligned = np.array([rets_i[idx_map[cd]]
                            for cd in common_dates if cd in idx_map])
        if len(aligned) == 0:
            continue
        cr      = np.exp(np.cumsum(aligned)) - 1.0
        w_label = f"{ticker} ({weights[tickers.index(ticker)]*100:.0f}%)"
        ax.plot(cr * 100, color=TICKER_PALETTE[ti], lw=0.9,
                alpha=0.55, label=w_label)
    ax.plot(cumret_port * 100, color=port_color, lw=2.5,
            label="Portfolio (weighted)", zorder=5)
    ax.axhline(0, color="#555", lw=0.6, ls="--")
    ax.set_ylabel("Cumulative return (%)", fontsize=9)
    ax.set_title(f"(a) Cumulative return  ·  {pname}",
                 fontsize=10, fontweight="semibold")
    ax.legend(fontsize=7.5, ncol=3, loc="upper left", framealpha=0.85)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda v, _: f"{v:.0f}%"))
    ax.tick_params(axis="x", labelbottom=False)

    # (b) H(q) ────────────────────────────────────────────────────────────
    ax = axes[1]
    for ti, ticker in enumerate(tickers):
        hq    = hq_dict[ticker]
        valid = np.isfinite(hq)
        hsl_i = hq_slope(Q_VALS, hq)
        ax.plot(Q_VALS[valid], hq[valid],
                color=TICKER_PALETTE[ti], lw=1.7,
                label=f"{ticker}  slope={hsl_i:.4f}")
    ax.axhline(0.5, color="#999", lw=0.8, ls="--", alpha=0.6,
               label="H=0.5 (RW)")
    ax.axhline(1.0, color="#66a", lw=0.7, ls=":",  alpha=0.5)
    ax.axvline(0,   color="#ccc", lw=0.5)
    ax.set_xlabel("q", fontsize=9)
    ax.set_ylabel("H(q)", fontsize=9)
    ax.set_title(
        "(b) H(q) — generalised Hurst spectrum  [q ∈ (−30,+30), step=5]\n"
        "Slope = multifractality  ·  level at q=2 = classical Hurst",
        fontsize=10, fontweight="semibold"
    )
    ax.legend(fontsize=7.5, ncol=1, loc="upper right", framealpha=0.85)

    # (c) τ(q) ────────────────────────────────────────────────────────────
    ax = axes[2]
    tau_mono = Q_VALS * 0.5 - 1.0
    ax.plot(Q_VALS, tau_mono, color="#bbb", lw=1.2, ls="--",
            label="Monofractal ref (H=0.5)", zorder=1)
    for ti, ticker in enumerate(tickers):
        tau_q = tau_dict[ticker]
        valid  = np.isfinite(tau_q)
        ax.plot(Q_VALS[valid], tau_q[valid],
                color=TICKER_PALETTE[ti], lw=1.7, label=ticker, zorder=2)
    ax.axhline(0, color="#555", lw=0.5, ls=":")
    ax.set_xlabel("q", fontsize=9)
    ax.set_ylabel("τ(q)", fontsize=9)
    ax.set_title(
        "(c) τ(q) = q·H(q) − 1  mass scaling exponent\n"
        "Deviation from dashed line = degree of multifractality",
        fontsize=10, fontweight="semibold"
    )
    ax.legend(fontsize=7.5, ncol=2, loc="upper left", framealpha=0.85)

    # (d) f(α) ────────────────────────────────────────────────────────────
    q0_idx = np.argmin(np.abs(Q_VALS))
    ax = axes[3]
    all_alpha_vals, all_Da_vals = [], []

    for ti, ticker in enumerate(tickers):
        alpha_i = alpha_dict[ticker]
        Da_i    = Da_dict[ticker]
        valid   = np.isfinite(alpha_i) & np.isfinite(Da_i)
        da_i    = delta_alpha(alpha_i, Da_i)
        ax.plot(alpha_i[valid], Da_i[valid],
                color=TICKER_PALETTE[ti], lw=1.7,
                label=f"{ticker}  Δα={da_i:.3f}")
        if valid[q0_idx]:
            ax.plot(alpha_i[q0_idx], Da_i[q0_idx], "o",
                    color=TICKER_PALETTE[ti], markersize=5,
                    markeredgecolor="white", markeredgewidth=0.7, zorder=5)
        all_alpha_vals.extend(alpha_i[valid].tolist())
        all_Da_vals.extend(Da_i[valid].tolist())

    ax.axhline(1.0, color="#999", lw=0.7, ls="--", alpha=0.5,
               label="f(α)=1 ref")
    if all_alpha_vals:
        a_lo = np.percentile(all_alpha_vals,  2)
        a_hi = np.percentile(all_alpha_vals, 98)
        d_lo = max(0, np.percentile(all_Da_vals,  2) - 0.05)
        d_hi = np.percentile(all_Da_vals, 98) + 0.05
        ax.set_xlim(a_lo - 0.02, a_hi + 0.02)
        ax.set_ylim(d_lo, min(d_hi, 1.4))
    ax.set_xlabel("α  (singularity strength)", fontsize=9)
    ax.set_ylabel("f(α)  (fractal dimension)", fontsize=9)
    ax.set_title(
        "(d) f(α) singularity spectrum  —  dots = q=0 apex\n"
        "Width Δα = multifractal heterogeneity  ·  apex ≈ box-counting dim",
        fontsize=10, fontweight="semibold"
    )
    ax.legend(fontsize=7.5, ncol=1, loc="upper right", framealpha=0.85)

    # (e) Rolling Rényi V(t) ───────────────────────────────────────────────
    ax = axes[4]
    xs_r = np.arange(len(Vs))
    ax.fill_between(xs_r, 0, Vs, color=port_color, alpha=0.30, zorder=1)
    ax.plot(xs_r, Vs, color=port_color, lw=1.5,
            label="V(t) = std(τ_R curvature residuals)", zorder=2)
    ax.axhline(mean_V, color=port_color, lw=1.1, ls="--", alpha=0.85,
               label=f"⟨V⟩ = {mean_V:.4f}", zorder=3)
    ax.set_ylabel("V(t)", fontsize=9)
    ax.set_title(
        "(e) Rolling Rényi volatility index V(t)\n"
        "VaR multiplier = 1 + V(t)  ·  no constants",
        fontsize=10, fontweight="semibold"
    )
    ax.legend(fontsize=8, loc="upper right", framealpha=0.85)
    ax.tick_params(axis="x", labelbottom=False)

    # (f) Rényi-adjusted VaR ──────────────────────────────────────────────
    ax = axes[5]
    xs_r = np.arange(len(bvars))
    ax.plot(xs_r, bvars, color="#888780", lw=1.1, ls="--",
            label="Standard VaR (99%)")
    ax.plot(xs_r, avars, color="#A32D2D", lw=1.8,
            label="Rényi-adjusted VaR")
    ax.fill_between(xs_r, bvars, avars,
                    color="#F09595", alpha=0.45, label="Hidden risk gap")
    ax.set_ylabel("VaR (%)", fontsize=9)
    ax.set_title(
        f"(f) VaR comparison  ·  ⟨gap⟩ = ${mean_gap:.1f}K "
        f"per ${POSITION_SIZE:,}",
        fontsize=10, fontweight="semibold"
    )
    ax.legend(fontsize=8, loc="upper right", framealpha=0.85)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda v, _: f"{v:.3f}%"))

    # ── NEW (g) F_q(s) log-log for all q values ───────────────────────────
    ax = axes[6]
    n_q = len(Q_VALS)
    colors_fq = [FQ_CMAP(i / (n_q - 1)) for i in range(n_q)]

    # Per-ticker lines: each asset gets a distinct linestyle so they're
    # distinguishable even though q-colour is shared across assets.
    ASSET_LINESTYLES = ["-", "--", "-.", ":", (0, (3,1,1,1))]
    ASSET_DASHCAPS   = ["butt", "butt", "butt", "round", "butt"]

    for ti, ticker in enumerate(tickers):
        scales_t = scales_dict[ticker]
        Fqs_t    = fqs_dict[ticker]
        if len(scales_t) == 0:
            continue
        ls = ASSET_LINESTYLES[ti % len(ASSET_LINESTYLES)]
        # Label only once per asset (use a proxy artist)
        proxy_plotted = False
        for j, (q, c) in enumerate(zip(Q_VALS, colors_fq)):
            row  = Fqs_t[j]
            mask = np.isfinite(row) & (row > 0)
            if mask.sum() < 2:
                continue
            lbl = ticker if (not proxy_plotted and q == Q_VALS[len(Q_VALS)//2]) else None
            ax.loglog(scales_t[mask], row[mask],
                      color=c, lw=0.8, alpha=0.45, linestyle=ls, label=lbl)
            if lbl:
                proxy_plotted = True

    # Per-asset legend (linestyle proxy patches)
    import matplotlib.lines as mlines
    asset_handles = [
        mlines.Line2D([], [], color="#777",
                      linestyle=ASSET_LINESTYLES[ti % len(ASSET_LINESTYLES)],
                      lw=1.5, label=ticker)
        for ti, ticker in enumerate(tickers)
    ]
    leg_assets = ax.legend(handles=asset_handles, fontsize=7.5,
                           loc="upper left", framealpha=0.85,
                           title="Asset (linestyle)", title_fontsize=7)
    ax.add_artist(leg_assets)   # keep it when we add the q legend below

    # Overlay portfolio-level F_q — run DFA on the weighted return series
    # compute port returns and run DFA on them for a clean portfolio Fq
    s_min_p = max(SCALE_MIN, POLY_ORD + 2)
    s_max_p = min(SCALE_MAX, len(port_r) // 4)
    scales_p, Fqs_p, _ = mfdfa(port_r, Q_VALS,
                                scale_min=s_min_p, scale_max=s_max_p)
    legend_handles = []
    for j, (q, c) in enumerate(zip(Q_VALS, colors_fq)):
        row  = Fqs_p[j]
        mask = np.isfinite(row) & (row > 0)
        if mask.sum() < 2:
            continue
        h, = ax.loglog(scales_p[mask], row[mask],
                       color=c, lw=2.0, alpha=0.95)
        legend_handles.append((h, f"q={int(q)}"))

    # Sparse legend (every other q)
    step  = max(1, len(legend_handles) // 8)
    shown = legend_handles[::step]
    ax.legend([h for h, _ in shown], [l for _, l in shown],
              fontsize=6.5, ncol=2, loc="lower right", framealpha=0.85)
    ax.set_xlabel("Scale  s  (days)", fontsize=9)
    ax.set_ylabel("$F_q(s)$", fontsize=9)
    ax.set_title(
        "(g) Fluctuation functions $F_q(s)$  —  all q values\n"
        "Thin: per-asset  ·  Bold: portfolio weighted return series\n"
        "Colour: blue=negative q  ·  red=positive q",
        fontsize=10, fontweight="semibold"
    )

    # Colorbar to show q mapping
    sm = plt.cm.ScalarMappable(cmap=FQ_CMAP,
                                norm=plt.Normalize(Q_VALS.min(), Q_VALS.max()))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label("q", fontsize=8)
    cbar.ax.tick_params(labelsize=7)

    # ── NEW (h) Symmetric F_{+q} / F_{-q} Pearson correlation ────────────
    ax = axes[7]
    mean_corr = np.full(len(Q_SYM), 0.0)
    count_corr = np.zeros(len(Q_SYM))

    for ti, ticker in enumerate(tickers):
        corr_vals = sym_corr_dict[ticker]
        valid     = np.isfinite(corr_vals)
        w_i       = weights[tickers.index(ticker)]
        ax.plot(Q_SYM[valid], corr_vals[valid],
                "o-", color=TICKER_PALETTE[ti], lw=1.5, markersize=3.5,
                alpha=0.70, label=f"{ticker} ({w_i*100:.0f}%)")
        mean_corr[valid]  += corr_vals[valid] * w_i
        count_corr[valid] += w_i

    # Portfolio-level weighted average
    valid_m = count_corr > 0
    mean_corr[valid_m] /= count_corr[valid_m]
    ax.plot(Q_SYM[valid_m], mean_corr[valid_m],
            "k-", lw=2.5, label="Portfolio weighted avg", zorder=10)
    ax.axhline(1.0, color="#999", lw=0.8, ls="--", alpha=0.5,
               label="r = 1 (perfect)")
    ax.axhline(0.0, color="#999", lw=0.6, ls=":",  alpha=0.4)

    # Shade the divergence from 1
    ax.fill_between(Q_SYM[valid_m], mean_corr[valid_m], 1.0,
                    color=port_color, alpha=0.18,
                    label="Asymmetry gap (portfolio)")

    # Annotate min correlation point
    if valid_m.sum() > 0:
        min_idx = int(np.argmin(mean_corr[valid_m]))
        q_min   = Q_SYM[valid_m][min_idx]
        r_min   = mean_corr[valid_m][min_idx]
        ax.annotate(f"min r={r_min:.4f}\nq={q_min:.0f}",
                    xy=(q_min, r_min), xytext=(q_min + 2, r_min - 0.02),
                    fontsize=7.5, color="#333",
                    arrowprops=dict(arrowstyle="->", color="#555", lw=0.8))

    ax.set_xlabel("|q|", fontsize=9)
    ax.set_ylabel("Pearson  r", fontsize=9)
    ax.set_xlim(1, 30)
    ax.set_xticks(Q_SYM[::3].astype(int))
    ax.set_title(
        "(h) Symmetric fluctuation correlation  r(F_{+q}, F_{-q})  vs |q|\n"
        "r→1: tail symmetry  ·  r↓: tail asymmetry / heavy-tail divergence",
        fontsize=10, fontweight="semibold"
    )
    ax.legend(fontsize=7.5, ncol=2, loc="lower left", framealpha=0.85)

    for ax in axes:
        ax.tick_params(direction="in", which="both", top=True, right=True)

    fname = (f"{OUT_DIR}/"
             f"{pname.replace(' ','_').replace('-','_')}_8panel.png")
    fig.savefig(fname, dpi=130, bbox_inches="tight")
    plt.close(fig)
    print(f"    Saved → {fname}")


# ─────────────────────────────────────────────────────────────────────────────
# CROSS-PORTFOLIO SCORECARD
# ─────────────────────────────────────────────────────────────────────────────
def _rank_score(values, lower_is_better=True):
    arr   = np.array(values, dtype=float)
    valid = np.isfinite(arr)
    out   = np.full(len(arr), 0.5)
    if valid.sum() < 2:
        return out
    mn, mx = arr[valid].min(), arr[valid].max()
    if mx == mn:
        return out
    normed     = (arr - mn) / (mx - mn)
    out[valid] = (1 - normed[valid]) if lower_is_better else normed[valid]
    return out


def plot_scorecard(scores):
    pnames  = list(scores.keys())
    metrics = ["Δα", "H(q) slope", "V̄", "VaR gap $K", "min r"]
    better  = [True,  True,         True, True,          False]

    # Risk-prioritised weights
    # Tail-risk metrics (V̄ + VaR gap) carry 70% of the total score;
    # fractal structure metrics (Δα, H slope, min r) share the remaining 30%.
    #   V̄            0.35  — rolling Rényi volatility (direct tail-risk proxy)
    #   VaR gap $K   0.35  — monetised hidden risk above standard VaR
    #   Δα           0.12  — singularity spectrum width (fractal heterogeneity)
    #   H(q) slope   0.06  — multifractality degree (scaling structure)
    #   min r        0.12  — tail asymmetry (F+q vs F-q correlation)
    WEIGHTS = np.array([0.12, 0.06, 0.35, 0.35, 0.12])   # order matches metrics

    raw       = np.array([[scores[p][m] for m in metrics] for p in pnames])
    ranked    = np.stack([_rank_score(raw[:, mi], lb)
                          for mi, lb in enumerate(better)], axis=1)
    composite = ranked @ WEIGHTS          # weighted dot product (sums to 1.0)
    best_idx  = int(np.argmax(composite))

    print(f"\n{'═'*86}")
    print(f"  CROSS-PORTFOLIO MULTIFRACTAL RISK SCORECARD  "
          f"(risk-prioritised weights)")
    print(f"  Weights: V̄=0.35  VaR gap=0.35  Δα=0.12  min r=0.12  H slope=0.06")
    print(f"{'═'*86}")
    hdr = (f"  {'Portfolio':<24}" +
           "".join(f"{m:>13}" for m in metrics) +
           f"  {'Score':>7}")
    print(hdr)
    print("  " + "─" * (len(hdr) - 2))
    for pi, pname in enumerate(pnames):
        row = f"  {pname:<24}"
        for m in metrics:
            row += f"{scores[pname][m]:>13.4f}"
        row += f"  {composite[pi]:>7.4f}"
        if pi == best_idx:
            row += "  ← BEST"
        print(row)
    print(f"\n  Winner: {pnames[best_idx]}")

    fig, axes = plt.subplots(1, 2, figsize=(17, 7))
    fig.suptitle(
        "Cross-Portfolio Multifractal Risk Scorecard  —  Risk-Prioritised Weights\n"
        "V̄ 35%  ·  VaR gap 35%  ·  Δα 12%  ·  min r 12%  ·  H slope 6%",
        fontsize=13, fontweight="bold"
    )

    # Left panel: grouped bars (weighted scores per metric, showing contribution)
    ax   = axes[0]
    x    = np.arange(len(pnames))
    w    = 0.14
    cmap = plt.cm.Set2

    # Bar heights = normalised score × weight (contribution to composite)
    metric_labels = [
        f"Δα  (×{WEIGHTS[0]:.2f})",
        f"H slope  (×{WEIGHTS[1]:.2f})",
        f"V̄  (×{WEIGHTS[2]:.2f})",
        f"VaR gap  (×{WEIGHTS[3]:.2f})",
        f"min r  (×{WEIGHTS[4]:.2f})",
    ]
    for mi, (m, lbl) in enumerate(zip(metrics, metric_labels)):
        contrib = ranked[:, mi] * WEIGHTS[mi]
        ax.bar(x + mi * w, contrib, w,
               label=lbl, color=cmap(mi / len(metrics)), alpha=0.88,
               edgecolor="#333", linewidth=0.5)

    ax.set_xticks(x + w * (len(metrics) - 1) / 2)
    ax.set_xticklabels(pnames, rotation=25, ha="right", fontsize=9)
    ax.set_ylabel("Weighted score contribution", fontsize=10)
    ax.set_title(
        "Per-metric weighted contributions\n"
        "(bar height = normalised score × metric weight)",
        fontsize=10
    )
    ax.legend(fontsize=8, loc="upper right", framealpha=0.92)
    ax.set_ylim(0, 0.42)
    ax.axhline(0.20, color="#aaa", lw=0.7, ls="--", alpha=0.5)

    # Right panel: composite bars
    ax     = axes[1]
    colors = [PORTFOLIO_COLORS.get(p, "#999") for p in pnames]
    bars   = ax.bar(pnames, composite, color=colors, alpha=0.88,
                    edgecolor="#333", linewidth=0.7)
    bars[best_idx].set_edgecolor("#FFD700")
    bars[best_idx].set_linewidth(3.5)
    for bar, val in zip(bars, composite):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.007,
                f"{val:.3f}", ha="center", va="bottom",
                fontsize=9.5, fontweight="bold")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Composite score  [1 = lowest risk / simplest scaling]",
                  fontsize=10)
    ax.set_title(
        f"Composite risk score  (risk-prioritised)\n"
        f"★  {pnames[best_idx]}  is the most risk-efficient portfolio",
        fontsize=11
    )
    ax.set_xticklabels(pnames, rotation=25, ha="right", fontsize=9)

    note = (
        "Metric weights (tail-risk dominant, 70%):  V̄=0.35  VaR gap=0.35  "
        "Δα=0.12  min r=0.12  H slope=0.06\n"
        "Portfolio construction  (all from same universe, Ledoit-Wolf Σ):\n"
        "  Min-Variance       Markowitz min-vol  (w'Σw minimised)\n"
        "  Max-Sharpe         Tangency portfolio  ((μ−rf)/σ maximised)\n"
        "  Risk-Parity        Equal Risk Contribution  (Maillard 2010)\n"
        "  Max-Diversification  DR = Σwᵢσᵢ / σ_p maximised  (Choueifong 2008)\n"
        "  Min-Correlation    Subset with lowest avg |ρᵢⱼ|  (equal-weight)\n"
        "  Max-Decorrelation  Weights minimising w'|C|w"
    )
    fig.text(0.5, -0.14, note, ha="center", fontsize=8.5,
             family="monospace", color="#333",
             bbox=dict(boxstyle="round,pad=0.6", fc="#f9f9f9", ec="#ccc"))

    plt.tight_layout()
    fname = f"{OUT_DIR}/scorecard.png"
    fig.savefig(fname, dpi=130, bbox_inches="tight")
    plt.close(fig)
    print(f"\n  Scorecard → {fname}")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":

    # 1. Download universe
    print("Pre-downloading universe …")
    for t in UNIVERSE:
        try:
            get_returns(t)
            print(f"  {t} ✓")
        except Exception as e:
            print(f"  {t} FAILED: {e}")

    # 2. Optimise portfolios
    print("\nOptimising portfolios …")
    portfolios = build_portfolios(UNIVERSE)

    # 3. MF analysis + plots
    scores = {}
    for pname, pdata in portfolios.items():
        plot_portfolio(pname, pdata["tickers"], pdata["weights"],
                       PORTFOLIO_COLORS.get(pname, "#999"), scores)

    # 4. Scorecard
    plot_scorecard(scores)
    print(f"\nAll outputs in:  {OUT_DIR}/")

Pre-downloading universe …
  AAPL ✓
  MSFT ✓
  NVDA ✓
  AMZN ✓
  META ✓
  TSLA ✓
  NFLX ✓
  AMD ✓
  JPM ✓
  GS ✓
  BAC ✓
  V ✓
  MA ✓
  JNJ ✓
  PG ✓
  KO ✓
  WMT ✓
  PEP ✓
  XOM ✓
  CVX ✓
  CAT ✓
  HON ✓
  GC=F ✓
  SI=F ✓
  CL=F ✓
  BTC-USD ✓
  ETH-USD ✓
  EWJ ✓
  EWZ ✓
  FXI ✓
  EWG ✓

Optimising portfolios …
Building universe return matrix …
  31 assets  ·  2111 common trading days

Running optimisers …
  Min-Variance             → ['GC=F', 'EWJ', 'JNJ', 'KO', 'PG']
                             weights=[0.427 0.188 0.17  0.12  0.094]
  Max-Sharpe               → ['GC=F', 'WMT', 'NVDA', 'CAT', 'TSLA']
                             weights=[0.408 0.291 0.153 0.127 0.021]
  Risk-Parity              → ['GC=F', 'JNJ', 'PG', 'WMT', 'KO']
                             weights=[0.334 0.179 0.17  0.161 0.156]
  Max-Diversification      → ['GC=F', 'JNJ', 'PG', 'WMT', 'CL=F']
                             weights=[0.358 0.211 0.149 0.142 0.141]
  Min-Correlation          → ['GS', 'PG', 'GC=F', 'CL